# Redes neuronales: clasificación, regresión y series de tiempo
**Kevin del Jesús González Maas** · Sistemas Inteligentes · Prof. Justino Ramírez Ortegón · FI-UAC

Ejecuta *Entorno de ejecución → Ejecutar todo*. Cada ejemplo descarga su dataset desde GitHub.

In [ ]:
!pip -q install numpy pandas scikit-learn matplotlib

In [ ]:
%%writefile comun.py
"""
comun.py - Utilidades compartidas por los tres ejemplos.
Autor: Kevin del Jesús González Maas | Sistemas Inteligentes | FI-UAC
"""
import os
import urllib.request
import matplotlib
matplotlib.use("Agg")                      # sin ventana: guarda directo a PNG
import matplotlib.pyplot as plt

RAIZ = os.path.dirname(os.path.abspath(__file__))
DIR_DATOS = os.path.join(RAIZ, "datos")
DIR_FIGS = os.path.join(RAIZ, "figuras")
SEMILLA = 42

# Paleta de ingeniería (sobria, apta para impresión)
AZUL, ROJO, VERDE, NARANJA, GRIS, NEGRO = ("#1F4E79", "#C0392B", "#2E7D32",
                                           "#E67E22", "#7F8C8D", "#212121")


def estilo_figuras():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200, "font.family": "DejaVu Sans",
        "font.size": 10, "axes.titlesize": 11.5, "axes.titleweight": "bold",
        "axes.labelsize": 10, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "--",
        "legend.frameon": False, "figure.facecolor": "white",
    })


def descargar(url, nombre):
    """Descarga el dataset a ./datos si aún no existe (caché local)."""
    os.makedirs(DIR_DATOS, exist_ok=True)
    ruta = os.path.join(DIR_DATOS, nombre)
    if not os.path.exists(ruta):
        print(f"[datos] Descargando {nombre} ...")
        urllib.request.urlretrieve(url, ruta)
    return ruta


def guardar(fig, nombre):
    os.makedirs(DIR_FIGS, exist_ok=True)
    ruta = os.path.join(DIR_FIGS, nombre)
    fig.savefig(ruta, bbox_inches="tight")
    plt.close(fig)
    print(f"[figura] {os.path.relpath(ruta, RAIZ)}")
    return ruta


def encabezado(titulo):
    linea = "=" * 72
    print(f"\n{linea}\n{titulo}\n{linea}")


In [ ]:
%%writefile ej1_clasificacion_churn.py
"""
EJEMPLO 1 - RED NEURONAL TIPO PATTERN RECOGNITION (CLASIFICACION)
Problema comercial: predecir la fuga de clientes (churn) de una empresa de telecomunicaciones.
Dataset : IBM Telco Customer Churn (7,043 clientes, 20 variables, clase Yes/No).
Modelo  : Perceptron multicapa (MLPClassifier, scikit-learn) con ajuste de hiperparametros.
Autor   : Kevin del Jesus Gonzalez Maas - Sistemas Inteligentes - FI-UAC
"""
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             precision_recall_curve, average_precision_score)
import comun as c

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
c.estilo_figuras()

# ---------------------------------------------------------------- 1. DATOS
c.encabezado("1. CARGA Y PREPROCESAMIENTO DE DATOS")
df = pd.read_csv(c.descargar(URL, "telco_churn.csv"))
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
y = (df["Churn"] == "Yes").astype(int).values
X = pd.get_dummies(df.drop(columns=["customerID", "Churn"]), drop_first=True).astype(float)
print(f"Registros: {len(df)} | Variables de entrada tras codificacion: {X.shape[1]}")
print(f"Clase positiva (abandona): {y.mean():.1%} | Clase negativa (permanece): {1-y.mean():.1%}")

# Division estratificada 70 / 15 / 15 (entrenamiento / validacion / prueba)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=c.SEMILLA)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=c.SEMILLA)
esc = StandardScaler().fit(X_tr)                 # el escalador solo ve entrenamiento
X_tr_s, X_va_s, X_te_s = esc.transform(X_tr), esc.transform(X_va), esc.transform(X_te)
print(f"Entrenamiento: {len(y_tr)} | Validacion: {len(y_va)} | Prueba: {len(y_te)}")

# El desbalance (26.5 % vs 73.5 %) se trata con el UMBRAL de decision (seccion 3),
# sin duplicar muestras: asi la validacion interna del early stopping no se contamina.
X_bal, y_bal = X_tr_s, y_tr

# ------------------------------------------------- 2. AJUSTE DE HIPERPARAMETROS
c.encabezado("2. AJUSTE DE HIPERPARAMETROS (busqueda en rejilla, criterio: AUC en validacion)")
rejilla = {"hidden_layer_sizes": [(8,), (16,), (32,), (32, 16)],
           "alpha": [1e-1, 1.0, 3.0, 10.0],
           "learning_rate_init": [1e-3, 5e-3]}
resultados = []
for h, a, lr in itertools.product(*rejilla.values()):
    m = MLPClassifier(hidden_layer_sizes=h, alpha=a, learning_rate_init=lr, activation="relu",
                      solver="adam", early_stopping=True, validation_fraction=0.15,
                      n_iter_no_change=12, max_iter=300, random_state=c.SEMILLA)
    m.fit(X_bal, y_bal)
    auc = roc_auc_score(y_va, m.predict_proba(X_va_s)[:, 1])
    resultados.append((auc, h, a, lr, m.n_iter_))
    print(f"  capas={str(h):<10} alpha={a:<6} lr={lr:<6} epocas={m.n_iter_:<3} AUC_val={auc:.4f}")
resultados.sort(key=lambda t: -t[0])
auc_mejor, h_opt, a_opt, lr_opt, _ = resultados[0]
print(f"\n>> Mejor configuracion: capas={h_opt}, alpha={a_opt}, lr={lr_opt}  (AUC_val={auc_mejor:.4f})")

# ------------------------------------------------------ 3. ENTRENAMIENTO FINAL
c.encabezado("3. ENTRENAMIENTO FINAL")
red = MLPClassifier(hidden_layer_sizes=h_opt, alpha=a_opt, learning_rate_init=lr_opt,
                    activation="relu", solver="adam", early_stopping=True,
                    validation_fraction=0.15, n_iter_no_change=12, max_iter=300,
                    random_state=c.SEMILLA).fit(X_bal, y_bal)
print(f"Arquitectura: {X.shape[1]} -> {' -> '.join(map(str, h_opt))} -> 1 (sigmoide)")
print(f"Epocas ejecutadas: {red.n_iter_} | Mejor puntaje de validacion interna: {red.best_validation_score_:.4f}")

# Umbral de decision optimo (maximiza F1 sobre validacion)
p_va = red.predict_proba(X_va_s)[:, 1]
umbrales = np.linspace(0.2, 0.85, 66)
f1s = [f1_score(y_va, p_va >= t) for t in umbrales]
t_opt = float(umbrales[int(np.argmax(f1s))])
print(f"Umbral de decision optimo (max F1 en validacion): {t_opt:.2f}")

# ------------------------------------------------------------ 4. EVALUACION
c.encabezado("4. EVALUACION SOBRE EL CONJUNTO DE PRUEBA (datos nunca vistos)")
p_te = red.predict_proba(X_te_s)[:, 1]
y_hat = (p_te >= t_opt).astype(int)
logit = LogisticRegression(max_iter=1000).fit(X_bal, y_bal)
p_log = logit.predict_proba(X_te_s)[:, 1]
f1s_log = [f1_score(y_va, logit.predict_proba(X_va_s)[:, 1] >= t) for t in umbrales]
t_log = float(umbrales[int(np.argmax(f1s_log))])
base = DummyClassifier(strategy="most_frequent").fit(X_tr_s, y_tr)

def fila(nombre, yh, p):
    return (nombre, accuracy_score(y_te, yh), precision_score(y_te, yh, zero_division=0),
            recall_score(y_te, yh), f1_score(y_te, yh), roc_auc_score(y_te, p) if p is not None else 0.5)

tabla = pd.DataFrame([fila("Clase mayoritaria (referencia)", base.predict(X_te_s), None),
                      fila("Regresion logistica (umbral propio)", (p_log >= t_log).astype(int), p_log),
                      fila("Red neuronal MLP (propuesta)", y_hat, p_te)],
                     columns=["Modelo", "Exactitud", "Precision", "Sensibilidad", "F1", "AUC"])
print(tabla.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
cm = confusion_matrix(y_te, y_hat)
tn, fp, fn, tp = cm.ravel()
print(f"\nMatriz de confusion  [VN={tn}  FP={fp}  FN={fn}  VP={tp}]")
print(f"Especificidad: {tn/(tn+fp):.4f} | Valor predictivo negativo: {tn/(tn+fn):.4f}")
print(f"AUC (red neuronal): {roc_auc_score(y_te, p_te):.4f} | Precision promedio (AP): {average_precision_score(y_te, p_te):.4f}")

# Impacto economico ilustrativo (supuestos explicitos, NO datos reales de la empresa)
costo_perder, costo_campana, exito = 600.0, 80.0, 0.30   # valor del cliente, costo por oferta, exito de retencion
def beneficio(tp_, fp_):  # se contacta a todo cliente marcado como "abandona"
    return tp_ * (exito * costo_perder - costo_campana) - fp_ * costo_campana
b_red = beneficio(tp, fp)
b_todos = beneficio(int(y_te.sum()), int((1 - y_te).sum()))
print(f"\nImpacto economico ilustrativo (cliente = ${costo_perder:.0f}, oferta = ${costo_campana:.0f}, exito de retencion = {exito:.0%}):")
print(f"  Campana guiada por la red neuronal : ${b_red:>9,.0f}")
print(f"  Campana a todos los clientes       : ${b_todos:>9,.0f}")
print(f"  Sin campana                        : ${0:>9,.0f}")

# ------------------------------------------------------------ 5. IMPORTANCIA
imp = permutation_importance(red, X_te_s, y_te, scoring="roc_auc", n_repeats=5,
                             random_state=c.SEMILLA)
orden = np.argsort(imp.importances_mean)[::-1][:10]
print("\nTop 10 variables mas influyentes (importancia por permutacion, caida de AUC):")
for i in orden:
    print(f"  {X.columns[i]:<42} {imp.importances_mean[i]:.4f}")

# ------------------------------------------------------------------ 6. FIGURAS
fig, ax = plt.subplots(2, 3, figsize=(16, 9.2))
fig.suptitle("Ejemplo 1 · Pattern Recognition: predicción de fuga de clientes (Telco Churn)",
             fontsize=14, fontweight="bold", y=1.0)
a = ax[0, 0]
a.plot(red.loss_curve_, color=c.AZUL, lw=2, label="Pérdida (entrenamiento)")
a.set_xlabel("Época"); a.set_ylabel("Entropía cruzada"); a.set_title("Curva de aprendizaje")
a2 = a.twinx(); a2.grid(False)
a2.plot(red.validation_scores_, color=c.NARANJA, lw=2, label="Exactitud (validación)")
a2.set_ylabel("Exactitud validación"); a2.spines["right"].set_visible(True)
h1, l1 = a.get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
a.legend(h1 + h2, l1 + l2, loc="center right")

a = ax[0, 1]
for p, nom, col in [(p_te, f"MLP  (AUC = {roc_auc_score(y_te, p_te):.3f})", c.AZUL),
                    (p_log, f"Reg. logística  (AUC = {roc_auc_score(y_te, p_log):.3f})", c.GRIS)]:
    fpr, tpr, _ = roc_curve(y_te, p); a.plot(fpr, tpr, lw=2.2, color=col, label=nom)
a.plot([0, 1], [0, 1], "--", color="k", lw=1, label="Azar")
a.set_xlabel("Tasa de falsos positivos"); a.set_ylabel("Tasa de verdaderos positivos")
a.set_title("Curva ROC (prueba)"); a.legend(loc="lower right")

a = ax[0, 2]
im = a.imshow(cm, cmap="Blues"); a.grid(False)
for (i, j), v in np.ndenumerate(cm):
    a.text(j, i, f"{v}\n({v/cm.sum():.1%})", ha="center", va="center", fontsize=12,
           color="white" if v > cm.max() / 2 else "black", fontweight="bold")
a.set_xticks([0, 1], ["Permanece", "Abandona"]); a.set_yticks([0, 1], ["Permanece", "Abandona"])
a.set_xlabel("Clase predicha"); a.set_ylabel("Clase real"); a.set_title(f"Matriz de confusión (umbral {t_opt:.2f})")

a = ax[1, 0]
a.plot(umbrales, f1s, color=c.VERDE, lw=2.2, label="F1 (validación)")
a.axvline(t_opt, color=c.ROJO, ls="--", label=f"Umbral óptimo = {t_opt:.2f}")
a.set_xlabel("Umbral de decisión"); a.set_ylabel("F1"); a.set_title("Selección del umbral"); a.legend()

a = ax[1, 1]
pr, rc, _ = precision_recall_curve(y_te, p_te)
a.plot(rc, pr, color=c.AZUL, lw=2.2, label=f"MLP  (AP = {average_precision_score(y_te, p_te):.3f})")
a.axhline(y_te.mean(), color="k", ls="--", lw=1, label="Referencia (prevalencia)")
a.set_xlabel("Sensibilidad (recall)"); a.set_ylabel("Precisión"); a.set_title("Curva Precisión–Sensibilidad"); a.legend()

a = ax[1, 2]
nombres = [X.columns[i].replace("_", " ")[:30] for i in orden][::-1]
a.barh(nombres, imp.importances_mean[orden][::-1], xerr=imp.importances_std[orden][::-1],
       color=c.AZUL, alpha=.9)
a.set_xlabel("Caída del AUC al permutar"); a.set_title("Variables más influyentes"); a.tick_params(axis="y", labelsize=8)
fig.tight_layout()
c.guardar(fig, "ej1_resultados.png")
print("\n[OK] Ejemplo 1 finalizado.")


In [ ]:
%%writefile ej2_regresion_concreto.py
"""
EJEMPLO 2 - RED NEURONAL TIPO FITTING (REGRESION)
Problema industrial: estimar la resistencia a compresion del concreto (MPa) a partir de su
dosificacion y edad de curado, evitando ensayos destructivos prolongados.
Dataset : Concrete Compressive Strength (UCI, I-Cheng Yeh, 1998) - 1,030 mezclas, 8 variables.
Modelo  : Perceptron multicapa (MLPRegressor, scikit-learn) con ajuste de hiperparametros.
Autor   : Kevin del Jesus Gonzalez Maas - Sistemas Inteligentes - FI-UAC
"""
import itertools, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import comun as c

warnings.filterwarnings('ignore')   # avisos de convergencia de L-BFGS (esperados)

URL = ("https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/"
       "master/concrete.csv")
c.estilo_figuras()
ETIQ = {"cement": "Cemento", "slag": "Escoria", "ash": "Ceniza volante", "water": "Agua",
        "superplastic": "Superplastificante", "coarseagg": "Agregado grueso",
        "fineagg": "Agregado fino", "age": "Edad (días)"}

# ---------------------------------------------------------------- 1. DATOS
c.encabezado("1. CARGA Y ANALISIS EXPLORATORIO")
df = pd.read_csv(c.descargar(URL, "concreto.csv"))
print(f"Mezclas: {len(df)} | Variables de entrada: 8 | Variable objetivo: strength (MPa)")
print(f"Datos faltantes: {int(df.isna().sum().sum())}")
print(df.describe().loc[["min", "mean", "max"]].T.round(2).to_string())
X, y = df.drop(columns="strength").values, df["strength"].values
nombres = list(df.columns[:-1])

# 15 % de prueba reservado y NO se toca hasta el final; el 85 % restante es "desarrollo".
X_dev, X_te, y_dev, y_te = train_test_split(X, y, test_size=0.15, random_state=c.SEMILLA)
print(f"\nDesarrollo (entrenamiento + validacion cruzada): {len(y_dev)} | Prueba (reservado): {len(y_te)}")

def fabricar(Xa, ya):
    """Normaliza con estadisticas del conjunto dado -> (escalador_x, escalador_y)"""
    return StandardScaler().fit(Xa), StandardScaler().fit(ya.reshape(-1, 1))

# ------------------------------------------------- 2. AJUSTE DE HIPERPARAMETROS
c.encabezado("2. AJUSTE DE HIPERPARAMETROS (validacion cruzada de 5 pliegues, criterio: RMSE)")
rejilla = {"hidden_layer_sizes": [(16,), (32,), (64,), (32, 16), (64, 32)],
           "activation": ["tanh", "relu"],
           "alpha": [1e-2, 1e-1, 1.0]}
kf = KFold(n_splits=5, shuffle=True, random_state=c.SEMILLA)
res, t0 = [], time.time()
for h, act, a in itertools.product(*rejilla.values()):
    rm = []
    for i_tr, i_va in kf.split(X_dev):
        ex_, ey_ = fabricar(X_dev[i_tr], y_dev[i_tr])
        m = MLPRegressor(hidden_layer_sizes=h, activation=act, alpha=a, solver="lbfgs",
                         max_iter=1500, random_state=c.SEMILLA)
        m.fit(ex_.transform(X_dev[i_tr]), ey_.transform(y_dev[i_tr].reshape(-1, 1)).ravel())
        p = ey_.inverse_transform(m.predict(ex_.transform(X_dev[i_va])).reshape(-1, 1)).ravel()
        rm.append(np.sqrt(mean_squared_error(y_dev[i_va], p)))
    res.append((np.mean(rm), np.std(rm), h, act, a))
res.sort(key=lambda r: r[0])
print(f"Se evaluaron {len(res)} configuraciones x 5 pliegues = {len(res)*5} entrenamientos ({time.time()-t0:.0f} s)")
print("Top 6 configuraciones:")
for rm, sd, h, act, a in res[:6]:
    print(f"  capas={str(h):<9} activ={act:<5} alpha={a:<5} RMSE_CV = {rm:.3f} +/- {sd:.3f} MPa")
print(f"Peor configuracion evaluada: RMSE_CV = {res[-1][0]:.3f} MPa")
cv_rmse, cv_sd, h_opt, act_opt, a_opt = res[0]
print(f"\n>> Mejor configuracion: capas={h_opt}, activacion={act_opt}, alpha={a_opt}")

# ------------------------------------------------------ 3. ENTRENAMIENTO FINAL
c.encabezado("3. ENTRENAMIENTO FINAL")
ex, ey = fabricar(X_dev, y_dev)
s = lambda a: ex.transform(a)
t = lambda v: ey.transform(v.reshape(-1, 1)).ravel()
inv = lambda v: ey.inverse_transform(np.asarray(v).reshape(-1, 1)).ravel()

# 3a. Curva de aprendizaje (Adam, particion 70/15 del total) para visualizar convergencia
X_a, X_v, y_a, y_v = train_test_split(X_dev, y_dev, test_size=0.1765, random_state=c.SEMILLA)
exa, eya = fabricar(X_a, y_a)
curva = MLPRegressor(hidden_layer_sizes=h_opt, activation=act_opt, alpha=a_opt, solver="adam",
                     learning_rate_init=3e-3, max_iter=1, warm_start=True, random_state=c.SEMILLA)
EPOCAS = 1500
perd_tr, perd_va = [], []
tt = lambda v: eya.transform(v.reshape(-1, 1)).ravel()
for ep in range(EPOCAS):
    curva.fit(exa.transform(X_a), tt(y_a))
    perd_tr.append(mean_squared_error(tt(y_a), curva.predict(exa.transform(X_a))))
    perd_va.append(mean_squared_error(tt(y_v), curva.predict(exa.transform(X_v))))
mejor_ep = int(np.argmin(perd_va)) + 1
print(f"Curva de aprendizaje (Adam, {len(y_a)} entrenamiento / {len(y_v)} validacion): "
      f"minimo de validacion en la epoca {mejor_ep} de {EPOCAS}")

# 3b. Comite (ensamble) de 5 redes con distintas inicializaciones, entrenado con todo "desarrollo"
SEMILLAS = [c.SEMILLA + k for k in range(5)]
comite = [MLPRegressor(hidden_layer_sizes=h_opt, activation=act_opt, alpha=a_opt, solver="lbfgs",
                       max_iter=3000, random_state=sd).fit(s(X_dev), t(y_dev)) for sd in SEMILLAS]
predecir = lambda Xn: inv(np.mean([m.predict(s(Xn)) for m in comite], axis=0))
red = comite[0]
print(f"Arquitectura de cada red: 8 -> {' -> '.join(map(str, h_opt))} -> 1 (lineal) | oculta: {act_opt}")
print(f"Parametros entrenables por red: {sum(w.size for w in red.coefs_) + sum(b.size for b in red.intercepts_)} "
      f"| redes en el comite: {len(comite)}")

# ------------------------------------------------------------ 4. EVALUACION
c.encabezado("4. EVALUACION")
lin = LinearRegression().fit(X_dev, y_dev)
def metricas(nombre, yr, yp):
    r = np.corrcoef(yr, yp)[0, 1]
    return (nombre, r, r2_score(yr, yp), np.sqrt(mean_squared_error(yr, yp)),
            mean_absolute_error(yr, yp), 100 * np.mean(np.abs((yr - yp) / yr)))
p_dev, p_te = predecir(X_dev), predecir(X_te)
p_una = inv(red.predict(s(X_te)))
tab = pd.DataFrame([metricas("Comite MLP - desarrollo", y_dev, p_dev),
                    metricas("Una sola red - prueba", y_te, p_una),
                    metricas("Comite MLP - PRUEBA", y_te, p_te),
                    metricas("Regresion lineal - prueba", y_te, lin.predict(X_te))],
                   columns=["Conjunto", "R", "R2", "RMSE (MPa)", "MAE (MPa)", "MAPE (%)"])
print(tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nRMSE en validacion cruzada (5 pliegues, una red): {cv_rmse:.3f} +/- {cv_sd:.3f} MPa")
err = y_te - p_te
print(f"Error en prueba: media = {err.mean():+.3f} MPa | desv. estandar = {err.std():.3f} MPa")
print(f"El 90 % de las predicciones cae a menos de {np.percentile(np.abs(err), 90):.2f} MPa del valor real")
print(f"Mejora del RMSE respecto a la regresion lineal: {100*(1-tab.iloc[2,3]/tab.iloc[3,3]):.1f} %")

# Importancia por permutacion expresada en MPa (aumento del RMSE al desordenar cada variable)
rng = np.random.RandomState(c.SEMILLA)
rmse0 = np.sqrt(mean_squared_error(y_te, p_te))
imp_m, imp_s = np.zeros(8), np.zeros(8)
for j in range(8):
    d = []
    for _ in range(30):
        Xp = X_te.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
        d.append(np.sqrt(mean_squared_error(y_te, predecir(Xp))) - rmse0)
    imp_m[j], imp_s[j] = np.mean(d), np.std(d)
orden = np.argsort(imp_m)[::-1]
print("\nImportancia de cada variable (aumento del RMSE al permutarla, en MPa):")
for i in orden:
    print(f"  {ETIQ[nombres[i]]:<20} +{imp_m[i]:.2f} MPa")

ej = pd.DataFrame([[350, 0, 0, 180, 0, 1000, 800, 28]], columns=nombres)
print(f"\nPrediccion de ejemplo -> cemento 350, agua 180, edad 28 d: {predecir(ej.values)[0]:.2f} MPa")

# ------------------------------------------------------------------ 5. FIGURAS
fig, ax = plt.subplots(2, 3, figsize=(16, 9.2))
fig.suptitle("Ejemplo 2 · Fitting: resistencia a la compresión del concreto (MPa)",
             fontsize=14, fontweight="bold", y=1.0)
a = ax[0, 0]
suave = lambda v, w=25: np.convolve(v, np.ones(w) / w, mode="valid")   # media movil (quita ruido de mini-lotes)
for v, col, nom in [(perd_tr, c.AZUL, "Entrenamiento"), (perd_va, c.NARANJA, "Validación")]:
    a.plot(range(1, EPOCAS + 1), v, color=col, lw=0.6, alpha=.25)
    a.plot(range(25, EPOCAS + 1), suave(v), color=col, lw=2.2, label=nom)
a.axvline(mejor_ep, color=c.ROJO, ls="--", label=f"Mínimo validación (época {mejor_ep})")
a.set_yscale("log"); a.set_xlabel("Época"); a.set_ylabel("MSE (datos normalizados)")
a.set_title("Rendimiento durante el entrenamiento"); a.legend()

a = ax[0, 1]
for yr, yp, col, nom in [(y_dev, p_dev, c.GRIS, "Desarrollo"), (y_te, p_te, c.AZUL, "Prueba")]:
    a.scatter(yr, yp, s=16, alpha=.55 if col == c.GRIS else .9, color=col,
              label=f"{nom} (R = {np.corrcoef(yr, yp)[0,1]:.3f})")
lim = [0, 85]; a.plot(lim, lim, "k--", lw=1, label="Ajuste ideal  y = x")
a.set_xlim(lim); a.set_ylim(lim); a.set_xlabel("Resistencia real (MPa)")
a.set_ylabel("Resistencia estimada (MPa)"); a.set_title("Regresión: salida vs. objetivo"); a.legend(loc="upper left")

a = ax[0, 2]
a.hist(err, bins=24, color=c.AZUL, alpha=.9, edgecolor="white")
a.axvline(0, color=c.ROJO, lw=2); a.set_xlabel("Error = real − estimado (MPa)")
a.set_ylabel("Mezclas"); a.set_title(f"Histograma de errores (σ = {err.std():.2f} MPa)")

a = ax[1, 0]
a.scatter(p_te, err, s=18, color=c.VERDE, alpha=.85)
a.axhline(0, color="k", lw=1); a.axhline(2 * err.std(), color=c.ROJO, ls="--", lw=1)
a.axhline(-2 * err.std(), color=c.ROJO, ls="--", lw=1, label="±2σ")
a.set_xlabel("Resistencia estimada (MPa)"); a.set_ylabel("Residuo (MPa)")
a.set_title("Residuos vs. valor estimado"); a.legend()

a = ax[1, 1]
a.barh([ETIQ[nombres[i]] for i in orden][::-1], imp_m[orden][::-1],
       xerr=imp_s[orden][::-1], color=c.AZUL)
a.set_xlabel("Aumento del RMSE al permutar (MPa)"); a.set_title("Importancia de variables")

a = ax[1, 2]                      # dependencia parcial: edad x cemento
base = np.tile(np.median(X_dev, axis=0), (60, 1))   # resto de ingredientes en su mediana
for cem, col in [(200, c.GRIS), (300, c.AZUL), (400, c.NARANJA), (500, c.ROJO)]:
    B = base.copy(); B[:, 0] = cem; B[:, 7] = np.geomspace(1, 90, 60)
    a.plot(B[:, 7], predecir(B), lw=2.2, color=col, label=f"Cemento = {cem} kg/m³")
a.set_xscale("log"); a.set_xlabel("Edad de curado (días, escala log)"); a.set_ylabel("Resistencia estimada (MPa)")
a.set_title("Curvas de curado estimadas (1–90 días)"); a.legend()
fig.tight_layout()
c.guardar(fig, "ej2_resultados.png")
print("\n[OK] Ejemplo 2 finalizado.")


In [ ]:
%%writefile ej3_series_tiempo_pm25.py
"""
EJEMPLO 3 - RED NEURONAL TIPO SERIES DE TIEMPO (NARX: autorregresiva con entradas exogenas)
Problema industrial/ambiental: pronosticar la concentracion de PM2.5 en Beijing 6 horas adelante
y emitir una alerta (clasificacion) cuando se superen 150 ug/m3.
Dataset : Beijing PM2.5 (UCI, Liang et al. 2015) - 43,824 registros horarios, 2010-2014.
Modelo  : MLP con ventana de 24 h (MLPRegressor, scikit-learn) + early stopping cronologico.
Autor   : Kevin del Jesus Gonzalez Maas - Sistemas Inteligentes - FI-UAC
"""
import copy, itertools, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error, roc_auc_score,
                             precision_score, recall_score, f1_score, confusion_matrix)
import comun as c

warnings.filterwarnings("ignore")
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pollution.csv"
VENTANA, H_PRINCIPAL, UMBRAL_ALERTA = 24, 6, 150.0     # horas de historia, horizonte, ug/m3
HORIZONTES = [1, 3, 6, 12, 24]
c.estilo_figuras()

# ---------------------------------------------------------------- 1. DATOS
c.encabezado("1. CARGA Y PREPROCESAMIENTO DE LA SERIE")
df = pd.read_csv(c.descargar(URL, "pm25_beijing.csv"), na_values="NA")
df["fecha"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.set_index("fecha")
faltante = df["pm2.5"].isna()
print(f"Registros horarios: {len(df)} ({df.index[0]:%Y-%m-%d} a {df.index[-1]:%Y-%m-%d})")
print(f"PM2.5 faltante: {faltante.sum()} horas ({faltante.mean():.1%}); se interpola SOLO como entrada,")
print("                 nunca se evalua ni se entrena contra valores interpolados.")
pm = df["pm2.5"].interpolate(limit=24, limit_direction="both")
viento = pd.get_dummies(df["cbwd"]).astype(float)                # NW, NE, SE, cv
F = pd.DataFrame({"pm": np.log1p(pm), "dewp": df["DEWP"], "temp": df["TEMP"],
                  "pres": df["PRES"], "iws": np.log1p(df["Iws"])}, index=df.index)
retardos = [F.shift(k).add_suffix(f"_t-{k}") for k in range(VENTANA)]          # t, t-1, ..., t-23
fija = pd.concat([viento.add_prefix("viento_"), df[["Is", "Ir"]],
                  pd.DataFrame({"h_sin": np.sin(2*np.pi*df.index.hour/24),
                                "h_cos": np.cos(2*np.pi*df.index.hour/24),
                                "d_sin": np.sin(2*np.pi*df.index.dayofyear/365.25),
                                "d_cos": np.cos(2*np.pi*df.index.dayofyear/365.25)}, index=df.index)], axis=1)
X_all = pd.concat(retardos + [fija], axis=1)
# Entradas para el baseline lineal (ARX): mismas variables pero PM2.5 y viento en escala real
F_lin = F.copy(); F_lin["pm"], F_lin["iws"] = pm, df["Iws"]
X_lin = pd.concat([F_lin.shift(k).add_suffix(f"_t-{k}") for k in range(VENTANA)] + [fija], axis=1)
print(f"Vector de entrada: {VENTANA} h x 5 variables + {fija.shape[1]} fijas = {X_all.shape[1]} entradas")

def armar(h):
    """Construye (X, y, pm_actual, fechas, X_lineal) para pronosticar PM2.5 en t+h."""
    y = df["pm2.5"].shift(-h)                                    # valor real observado en t+h
    ok = X_all.notna().all(axis=1) & y.notna() & (~faltante) & (~faltante.shift(-h, fill_value=True))
    return X_all[ok].values, y[ok].values, df["pm2.5"][ok].values, X_all.index[ok], X_lin[ok].values

def particion(n):
    return int(0.70 * n), int(0.85 * n)                          # cronologica 70/15/15

def escalar(Xtr, *otros):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    return [(a - mu) / sd for a in (Xtr, *otros)]

# --------------------------------------- 2. ENTRENAMIENTO CON EARLY STOPPING CRONOLOGICO
def entrenar(Xtr, ytr, Xva, yva, h, alpha, lr, semilla=c.SEMILLA, max_ep=150, paciencia=12):
    """Adam por epocas; se conserva la red con menor error en validacion (que es POSTERIOR al entrenamiento)."""
    m = MLPRegressor(hidden_layer_sizes=h, alpha=alpha, learning_rate_init=lr, batch_size=256,
                     solver="adam", random_state=semilla)
    mejor, w, hist_tr, hist_va, sin_mejora = np.inf, None, [], [], 0
    for ep in range(max_ep):
        m.partial_fit(Xtr, ytr)
        e_tr, e_va = mean_squared_error(ytr, m.predict(Xtr)), mean_squared_error(yva, m.predict(Xva))
        hist_tr.append(e_tr); hist_va.append(e_va)
        if e_va < mejor - 1e-5:
            mejor, sin_mejora = e_va, 0
            w = (copy.deepcopy(m.coefs_), copy.deepcopy(m.intercepts_))
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia: break
    m.coefs_, m.intercepts_ = w
    return m, np.array(hist_tr), np.array(hist_va), int(np.argmin(hist_va)) + 1

def preparar(h):
    X, y, pm_act, fechas, XL = armar(h)
    i1, i2 = particion(len(y))
    Xtr, Xva, Xte = escalar(X[:i1], X[i1:i2], X[i2:])
    Ltr, Lva, Lte = escalar(XL[:i1], XL[i1:i2], XL[i2:])
    return dict(Xtr=Xtr, Xva=Xva, Xte=Xte, Ltr=Ltr, Lva=Lva, Lte=Lte,
                ytr=y[:i1], yva=y[i1:i2], yte=y[i2:],
                pm_tr=pm_act[:i1], pm_va=pm_act[i1:i2], pm_te=pm_act[i2:],
                f_te=fechas[i2:], fechas=fechas, i1=i1, i2=i2, y=y)

def ajustar(D, h, alpha, lr, forma):
    """Entrena la red. forma='directa': y(t+h) = red(x);  forma='residual': y(t+h) = PM2.5(t) + red(x).
    Devuelve el predictor en ug/m3: pred(X, pm_actual)."""
    base = (lambda pm: pm) if forma == "residual" else (lambda pm: 0.0 * pm)
    m, htr, hva, ep = entrenar(D["Xtr"], (D["ytr"] - base(D["pm_tr"])) / 100.0,
                               D["Xva"], (D["yva"] - base(D["pm_va"])) / 100.0, h, alpha, lr)
    return m, (lambda Xs, pm: np.maximum(m.predict(Xs) * 100.0 + base(pm), 0.0)), htr, hva, ep

def baseline_arx(Dh):
    """Autorregresivo lineal con entradas exogenas (ARX): Ridge sobre las mismas 24 h en escala real."""
    r = Ridge(alpha=10.0).fit(Dh["Ltr"], Dh["ytr"])
    return lambda Lx: np.maximum(r.predict(Lx), 0.0)

rmse = lambda y, p: float(np.sqrt(mean_squared_error(y, p)))
D = preparar(H_PRINCIPAL)
print(f"\nParticion cronologica (horizonte {H_PRINCIPAL} h): entrenamiento {D['i1']} "
      f"({D['fechas'][0]:%Y-%m-%d} a {D['fechas'][D['i1']-1]:%Y-%m-%d})")
print(f"                                   validacion    {D['i2']-D['i1']} "
      f"({D['fechas'][D['i1']]:%Y-%m-%d} a {D['fechas'][D['i2']-1]:%Y-%m-%d})")
print(f"                                   prueba        {len(D['yte'])} "
      f"({D['fechas'][D['i2']]:%Y-%m-%d} a {D['fechas'][-1]:%Y-%m-%d})")

# ------------------------------------------------- 2. AJUSTE DE HIPERPARAMETROS
c.encabezado("2. AJUSTE DE HIPERPARAMETROS (criterio: RMSE en validacion, ug/m3)")
rejilla = {"forma": ["directa", "residual"], "hidden_layer_sizes": [(32,), (64,), (64, 32)], "alpha": [0.1, 1.0, 3.0]}
res, t0 = [], time.time()
for forma, h, a in itertools.product(*rejilla.values()):
    m, pred, htr_, hva_, ep = ajustar(D, h, a, 3e-3, forma)
    rv = rmse(D["yva"], pred(D["Xva"], D["pm_va"]))
    res.append((rv, forma, h, a, ep))
    print(f"  forma={forma:<9} capas={str(h):<9} alpha={a:<4} mejor_epoca={ep:<3} RMSE_val={rv:.2f}")
res.sort(key=lambda r: r[0])
_, forma_opt, h_opt, a_opt, _ = res[0]
lr_opt = 3e-3
print(f"({len(res)} configuraciones, {time.time()-t0:.0f} s)")
print(f"RMSE de validacion de referencia: persistencia = {rmse(D['yva'], D['pm_va']):.2f} | "
      f"ARX lineal = {rmse(D['yva'], baseline_arx(D)(D['Lva'])):.2f}")
print(f">> Mejor configuracion: forma={forma_opt}, capas={h_opt}, alpha={a_opt}, lr={lr_opt}")

# ------------------------------------------------------ 3. ENTRENAMIENTO FINAL
c.encabezado("3. ENTRENAMIENTO FINAL (horizonte principal)")
red, pred_nn, htr, hva, mejor_ep = ajustar(D, h_opt, a_opt, lr_opt, forma_opt)
print(f"Arquitectura: {D['Xtr'].shape[1]} -> {' -> '.join(map(str, h_opt))} -> 1 | "
      f"parametros: {sum(w.size for w in red.coefs_) + sum(b.size for b in red.intercepts_)}")
print(f"Epocas entrenadas: {len(htr)} | mejor epoca (validacion): {mejor_ep}")

# ------------------------------------------------------------ 4. EVALUACION
c.encabezado(f"4. EVALUACION EN PRUEBA - PRONOSTICO A {H_PRINCIPAL} HORAS")
def mets(nombre, y, p):
    return (nombre, r2_score(y, p), rmse(y, p), mean_absolute_error(y, p), np.corrcoef(y, p)[0, 1])
p_nn, p_ar, p_pers = pred_nn(D["Xte"], D["pm_te"]), baseline_arx(D)(D["Lte"]), D["pm_te"]
tab = pd.DataFrame([mets("Persistencia (PM2.5 actual)", D["yte"], p_pers),
                    mets("Autorregresivo lineal ARX", D["yte"], p_ar),
                    mets("Red neuronal NARX (propuesta)", D["yte"], p_nn)],
                   columns=["Modelo", "R2", "RMSE", "MAE", "R"])
print(tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
rm = tab["RMSE"].values
print(f"\nHabilidad respecto a la persistencia (1 - RMSE/RMSE_pers): NARX = {1-rm[2]/rm[0]:+.1%} | "
      f"ARX = {1-rm[1]/rm[0]:+.1%}")
print(f"Mejora de la red NARX sobre el ARX lineal: {1-rm[2]/rm[1]:+.1%} en RMSE")
err = D["yte"] - p_nn
print(f"Error NARX: media = {err.mean():+.2f} ug/m3 | desviacion estandar = {err.std():.2f} ug/m3")

# Barrido de horizontes (forma y alpha se re-sintonizan en validacion para cada horizonte)
print("\nRMSE de prueba (ug/m3) por horizonte de pronostico (forma y alpha re-sintonizados en validacion):")
print(f"  {'h':>3} | {'forma':>8} | {'alpha':>5} | {'Persistencia':>12} | {'ARX':>7} | {'NARX':>7} | {'R2 NARX':>8}")
barrido = []
for h in HORIZONTES:
    Dh = preparar(h)
    cand = []
    for forma in ["directa", "residual"]:
        for a in [0.1, 1.0, 10.0]:
            mh, ph, *_ = ajustar(Dh, h_opt, a, lr_opt, forma)
            cand.append((rmse(Dh["yva"], ph(Dh["Xva"], Dh["pm_va"])), forma, a, ph))
    _, f_h, a_h, ph = min(cand, key=lambda t: t[0])
    pn, pa = ph(Dh["Xte"], Dh["pm_te"]), baseline_arx(Dh)(Dh["Lte"])
    fila = (h, rmse(Dh["yte"], Dh["pm_te"]), rmse(Dh["yte"], pa), rmse(Dh["yte"], pn), r2_score(Dh["yte"], pn))
    barrido.append(fila)
    print(f"  {h:>3} | {f_h:>8} | {a_h:>5} | {fila[1]:>12.2f} | {fila[2]:>7.2f} | {fila[3]:>7.2f} | {fila[4]:>8.4f}")
barrido = np.array(barrido)

# Clasificacion derivada: alerta si PM2.5(t+h) > 150 ug/m3 (umbral del modelo calibrado en validacion)
print(f"\nSistema de ALERTA (PM2.5 en t+{H_PRINCIPAL} h > {UMBRAL_ALERTA:.0f} ug/m3) - clasificacion binaria:")
real, real_va = (D["yte"] > UMBRAL_ALERTA).astype(int), (D["yva"] > UMBRAL_ALERTA).astype(int)
print(f"  Horas de prueba con alerta real: {real.sum()} de {len(real)} ({real.mean():.1%})")
cand_u = np.arange(60, 201, 2.5)
def umbral_opt(p_val):
    return float(cand_u[int(np.argmax([f1_score(real_va, p_val > u) for u in cand_u]))])
t_nn, t_pers = umbral_opt(pred_nn(D["Xva"], D["pm_va"])), umbral_opt(D["pm_va"])
print(f"  Umbral de pronostico calibrado en validacion (max F1): NARX = {t_nn:.1f} | persistencia = {t_pers:.1f} ug/m3")
for nom, p, u in [("Persistencia", p_pers, t_pers), ("NARX", p_nn, t_nn)]:
    pred = (p > u).astype(int)
    print(f"  {nom:<13} exactitud={np.mean(pred==real):.4f} precision={precision_score(real, pred):.4f} "
          f"sensibilidad={recall_score(real, pred):.4f} F1={f1_score(real, pred):.4f} AUC={roc_auc_score(real, p):.4f}")
cm = confusion_matrix(real, (p_nn > t_nn).astype(int))
print(f"  Matriz de confusion NARX [VN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} VP={cm[1,1]}]")

# Autocorrelacion del error (diagnostico estilo MATLAB)
e0 = err - err.mean()
ac = np.array([np.sum(e0[:len(e0)-k] * e0[k:]) for k in range(49)]) / np.sum(e0**2)
lim = 1.96 / np.sqrt(len(e0))

# ------------------------------------------------------------------ 5. FIGURAS
# 5a. Vista general de los datos
fig, ax = plt.subplots(1, 2, figsize=(16, 4.6), gridspec_kw={"width_ratios": [2.2, 1]})
fig.suptitle("Ejemplo 3 · Series de tiempo: exploración del conjunto de datos PM2.5 (Beijing, 2010–2014)",
             fontsize=14, fontweight="bold", y=1.03)
f_all, y_all = D["fechas"], D["y"]
a = ax[0]
for i0, i1_, col, nom in [(0, D["i1"], c.AZUL, "Entrenamiento (70 %)"), (D["i1"], D["i2"], c.NARANJA, "Validación (15 %)"),
                          (D["i2"], len(y_all), c.VERDE, "Prueba (15 %)")]:
    a.plot(f_all[i0:i1_], y_all[i0:i1_], lw=.5, color=col, label=nom)
a.axhline(UMBRAL_ALERTA, color=c.ROJO, ls="--", lw=1.2, label=f"Umbral de alerta ({UMBRAL_ALERTA:.0f} µg/m³)")
a.set_ylabel("PM2.5 (µg/m³)"); a.set_title("Serie horaria y partición cronológica"); a.legend(ncol=4, loc="upper center")
a = ax[1]
mes = pd.Series(y_all, index=f_all).groupby(f_all.month)
a.boxplot([g.values for _, g in mes], showfliers=False, patch_artist=True,
          boxprops=dict(facecolor="#BBD3EA"), medianprops=dict(color=c.ROJO, lw=2))
a.set_xticks(range(1, 13), ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
a.set_xlabel("Mes"); a.set_ylabel("PM2.5 (µg/m³)"); a.set_title("Estacionalidad (temporada de calefacción)")
fig.tight_layout(); c.guardar(fig, "ej3_datos.png")

# 5b. Resultados
fig, ax = plt.subplots(2, 3, figsize=(16, 9.2))
fig.suptitle(f"Ejemplo 3 · Series de tiempo (NARX): pronóstico de PM2.5 a {H_PRINCIPAL} horas",
             fontsize=14, fontweight="bold", y=1.0)
a = ax[0, 0]
a.plot(range(1, len(htr)+1), htr, color=c.AZUL, lw=2, label="Entrenamiento")
a.plot(range(1, len(hva)+1), hva, color=c.NARANJA, lw=2, label="Validación")
a.axvline(mejor_ep, color=c.ROJO, ls="--", label=f"Mejor época = {mejor_ep}")
a.set_xlabel("Época"); a.set_ylabel("MSE (escala normalizada)"); a.set_title("Rendimiento durante el entrenamiento"); a.legend()

a = ax[0, 1]                                         # ventana de 14 dias con el episodio mas severo
ventana = 24 * 14
pico = int(np.argmax(pd.Series(D["yte"]).rolling(ventana).max().values))
ini = max(0, min(pico - ventana // 2, len(D["yte"]) - ventana)); sl = slice(ini, ini + ventana)
a.fill_between(D["f_te"][sl], 0, D["yte"][sl], color="#D6DBDF", label="Real")
a.plot(D["f_te"][sl], p_nn[sl], color=c.AZUL, lw=1.8, label=f"NARX (t+{H_PRINCIPAL} h)")
a.plot(D["f_te"][sl], p_pers[sl], color=c.NARANJA, lw=1, alpha=.8, label="Persistencia")
a.axhline(UMBRAL_ALERTA, color=c.ROJO, ls="--", lw=1)
a.set_ylabel("PM2.5 (µg/m³)"); a.set_title("Pronóstico vs. real (14 días de prueba)"); a.legend(loc="upper right")
a.tick_params(axis="x", rotation=25)

a = ax[0, 2]
hb = a.hexbin(D["yte"], p_nn, gridsize=45, bins="log", cmap="Blues", mincnt=1, extent=(0, 500, 0, 500))
m_ = 500; a.plot([0, m_], [0, m_], "k--", lw=1, label="Ajuste ideal")
a.set_xlim(0, m_); a.set_ylim(0, m_); a.set_xlabel("PM2.5 real (µg/m³)"); a.set_ylabel("PM2.5 pronosticado (µg/m³)")
a.set_title(f"Regresión (R = {np.corrcoef(D['yte'], p_nn)[0,1]:.3f}, R² = {r2_score(D['yte'], p_nn):.3f})")
a.legend(loc="upper left"); plt.colorbar(hb, ax=a, label="Horas (log)")

a = ax[1, 0]
for j, nom, col, mk in [(1, "Persistencia", c.NARANJA, "s"), (2, "ARX lineal", c.GRIS, "^"), (3, "NARX (MLP)", c.AZUL, "o")]:
    a.plot(barrido[:, 0], barrido[:, j], marker=mk, lw=2.2, color=col, label=nom)
a.set_xlabel("Horizonte de pronóstico (h)"); a.set_ylabel("RMSE (µg/m³)"); a.set_title("Error vs. horizonte"); a.legend()

a = ax[1, 1]
a.bar(range(49), ac, color=c.AZUL, width=.7)
a.axhline(lim, color=c.ROJO, ls="--"); a.axhline(-lim, color=c.ROJO, ls="--", label="Límites de confianza 95 %")
a.set_xlabel("Retardo (h)"); a.set_ylabel("Autocorrelación"); a.set_title("Autocorrelación del error"); a.legend()

a = ax[1, 2]
a.imshow(cm, cmap="Blues"); a.grid(False)
for (i, j), v in np.ndenumerate(cm):
    a.text(j, i, f"{v}\n({v/cm.sum():.1%})", ha="center", va="center", fontsize=12, fontweight="bold",
           color="white" if v > cm.max() / 2 else "black")
a.set_xticks([0, 1], ["Sin alerta", "Alerta"]); a.set_yticks([0, 1], ["Sin alerta", "Alerta"])
a.set_xlabel("Predicho"); a.set_ylabel("Real")
a.set_title(f"Alerta > {UMBRAL_ALERTA:.0f} µg/m³  (AUC = {roc_auc_score(real, p_nn):.3f})")
fig.tight_layout(); c.guardar(fig, "ej3_resultados.png")
print("\n[OK] Ejemplo 3 finalizado.")


## Ejemplo 1 · Pattern Recognition (fuga de clientes)

In [ ]:
!python -u ej1_clasificacion_churn.py

In [ ]:
from IPython.display import Image, display
display(Image('figuras/ej1_resultados.png'))

## Ejemplo 2 · Fitting (resistencia del concreto)

In [ ]:
!python -u ej2_regresion_concreto.py

In [ ]:
from IPython.display import Image, display
display(Image('figuras/ej2_resultados.png'))

## Ejemplo 3 · Series de tiempo NARX (PM2.5)

In [ ]:
!python -u ej3_series_tiempo_pm25.py

In [ ]:
from IPython.display import Image, display
display(Image('figuras/ej3_resultados.png'))